# 04 · Query understanding and search

**Purpose.** Define the single search interface that the evaluation (05) and the demo (06) both call:

```python
search(query, system="baseline" | "improved" | <ablation>, k=5, snapshot_date=None)   # None = latest snapshot
```

**Inputs:** the indexes from 02, the condition-fit functions from 03 (both loaded with `%run`, nothing copied), `data/resources/gazetteer.csv`, `data/resources/regions.csv`, and a snapshot.

**Outputs:** the functions `parse_query`, `search`, `search_verbose`; when run directly, a side-by-side comparison of example queries.

| System | Query understanding | Synonym expansion | Condition fit | Notes |
|---|---|---|---|---|
| `baseline` | – | – | – | BM25 over profiles |
| `improved` | ✓ | ✓ | ✓ | full system |
| `improved_no_cf` | ✓ | ✓ | – | ablation: text only |
| `improved_no_qu` | – | ✓ | ✓ | ablation: defaults for place/time/skill |
| `improved_no_syn` | ✓ | – | ✓ | ablation: no jargon expansion |
| `improved_hybrid` | ✓ | ✓ | ✓ | BM25 + dense (RRF); only if a dense index exists |
| `improved_oracle` | ✓ | ✓ | ✓ (observed) | evaluation only: conditions from buoy observations = upper bound |

In [ ]:
# ---- Configuration ----
import os
from pathlib import Path


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "requirements.txt").is_file():
            return folder
    raise FileNotFoundError("Project root not found. Open the project folder in VS Code (File > Open Folder).")


ROOT = find_project_root()
os.chdir(ROOT / "notebooks")             # so the relative %run paths work whatever VS Code's working folder is

# Load 02 (analyser, index loaders) and 03 (condition fit) as libraries: their build/demo cells are skipped.
_outer_library_mode = os.environ.get("SURF_IR_LIBRARY_MODE", "0")
os.environ["SURF_IR_LIBRARY_MODE"] = "1"
%run ./02_indexing.ipynb
%run ./03_surfability.ipynb
os.environ["SURF_IR_LIBRARY_MODE"] = _outer_library_mode
LIBRARY_MODE = _outer_library_mode == "1"   # True when 05/06 load this notebook

GAZETTEER_FILE = ROOT / "data" / "resources" / "gazetteer.csv"
REGIONS_FILE = ROOT / "data" / "resources" / "regions.csv"

DEFAULT_K = 5
DEFAULT_SKILL = "intermediate"          # used when the query names no skill level (shown to the user)
DEFAULT_WINDOW_HOURS = 48               # used when the query names no time
NEAR_RADIUS_KM = 40                     # "near Kaohsiung" = within this distance. [TEAM] decide
FUZZY_THRESHOLD = 88                    # rapidfuzz ratio for place names (catches 'Kaohsuing', 'Taidong')
ALPHA = 0.5                             # weight of text relevance vs condition fit (tune on dev queries in 05)
ALPHA_NO_CONDITION_INTENT = 0.85        # queries about attributes only ("reef break with showers") lean on text
SYNONYM_DEFAULT_WEIGHT = 0.5
SAFETY_DEMOTE = True                    # spots unsafe for the user's level go to the bottom with a warning
RRF_K = 60                              # reciprocal rank fusion constant (Cormack et al., 2009)
SAMPLE_N = None                         # not used in this notebook (sampling of queries happens in 05)
SEED = 42

## 1. Load indexes and resources
We check that the index matches the corpus. A spot added to `spots.csv` but not re-indexed would otherwise be invisible to the baseline but visible to the condition scorer, an unfair and confusing comparison.

In [ ]:
import re
import time
from dataclasses import dataclass, field
from html import escape

import dateparser
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from rapidfuzz import fuzz, process

BM25_INDEX = load_bm25_index()
SYNONYMS = load_synonyms()
DENSE = load_dense_index()
GAZETTEER = pd.read_csv(GAZETTEER_FILE).fillna("")
REGION_PHRASES = pd.read_csv(REGIONS_FILE).fillna("")

_missing = set(SPOTS) ^ set(BM25_INDEX["doc_ids"])
if _missing:
    raise RuntimeError(f"Index and corpus differ for {sorted(_missing)}. Re-run 01 and 02.")
DOC_IDS = BM25_INDEX["doc_ids"]
SPOT_LAT = np.array([SPOTS[d]["lat"] for d in DOC_IDS], dtype=float)
SPOT_LON = np.array([SPOTS[d]["lon"] for d in DOC_IDS], dtype=float)
SPOT_REGION = np.array([SPOTS[d]["region"] for d in DOC_IDS])
if not LIBRARY_MODE:
    print(f"{len(DOC_IDS)} spots indexed; {len(SYNONYMS)} synonym entries; dense index: {'yes' if DENSE else 'no'}; "
          f"{len(GAZETTEER)} gazetteer places; snapshots: {list_snapshots()}")

## 2. Query understanding
We extract four things from the query, using transparent rules rather than a trained model. With ~50 test queries we could not train or validate a model, and rules can be inspected and cited in the report:

1. **Location:** a town or spot name (fuzzy-matched against the gazetteer, so *Gaoxiong* and *Kaohsuing* both find Kaohsiung) → a radius filter; or a region phrase (*east coast*, *down south*) → a region filter.
2. **Skill level:** lexicon (*first time*, *newbie*, *learner* → beginner).
3. **Time window:** *today, tomorrow, this weekend, Saturday, next 3 days, morning, dawn patrol, after work*, explicit dates. All are resolved relative to the **snapshot's reference time**, not the computer clock, so evaluation runs are reproducible.
4. **Condition intent:** does the user care about current conditions at all? *"Reef break with showers near Taitung"* is a pure attribute query; *"glassy waves tomorrow"* is not.

Anything not stated falls back to a visible default (all of Taiwan, next 48 h, intermediate).

In [ ]:
@dataclass
class ParsedQuery:
    raw: str
    text: str
    place: str = None
    place_latlon: tuple = None
    regions: list = field(default_factory=list)
    named_spots: list = field(default_factory=list)
    skill: str = DEFAULT_SKILL
    skill_from_query: bool = False
    windows: list = field(default_factory=list)          # list of (start, end) timestamps
    time_label: str = ""
    time_from_query: bool = False
    condition_intent: bool = False
    notes: list = field(default_factory=list)


SKILL_PATTERNS = {
    "beginner": r"\b(beginners?|beginner friendly|novices?|learners?|newbies?|first timers?|first time|learn(ing)? to surf|lessons?|never surfed)\b",
    "intermediate": r"\b(intermediates?|improvers?)\b",
    "advanced": r"\b(advanced|experts?|experienced|pros?)\b",
}
CONDITION_PATTERN = r"\b(waves?|swell|conditions?|glassy|offshore|clean|best|good|firing|pumping|forecast|surfable|session|surf (now|today|tomorrow))\b"
PARTS_OF_DAY = [("dawn patrol", 5, 8), ("early morning", 5, 8), ("dawn", 5, 8), ("sunrise", 5, 8),
                ("late afternoon", 15, 18), ("afternoon", 12, 18), ("morning", 6, 11), ("midday", 11, 14),
                ("noon", 11, 14), ("lunchtime", 11, 14), ("after work", 16, 19), ("evening", 16, 19), ("sunset", 16, 19)]
WEEKDAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
MONTH_PATTERN = r"\b(\d{1,2} (jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*|(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]* \d{1,2}|\d{4} \d{1,2} \d{1,2}|\d{1,2} \d{1,2})\b"


def normalize_query(q):
    return " ".join(tokenize(q))


def _place_entries():
    """Everything a location mention can match: towns from the gazetteer, region phrases, spot names."""
    entries = []
    for r in GAZETTEER.itertuples():
        for name in [r.place, *[v for v in r.variants.split(";") if v.strip()]]:
            entries.append({"name": normalize_query(name), "kind": "place", "key": r.place, "lat": r.lat, "lon": r.lon})
    for sid, s in SPOTS.items():
        for name in [s["name"], *(s.get("alt_names") or [])]:
            entries.append({"name": normalize_query(name), "kind": "spot", "key": sid, "lat": s["lat"], "lon": s["lon"]})
    return entries


PLACE_ENTRIES = _place_entries()
PLACE_NAMES = [e["name"] for e in PLACE_ENTRIES]


def parse_location(text, pq):
    tokens = text.split()
    used = set()
    for r in REGION_PHRASES.sort_values("phrase", key=lambda s: -s.str.len()).itertuples():   # longest phrase first
        phrase = normalize_query(r.phrase)
        m = re.search(rf"\b{re.escape(phrase)}\b", text)
        if m and not any(i in used for i in range(m.start(), m.end())):
            used.update(range(m.start(), m.end()))
            regions = [x.strip() for x in r.regions.split(";") if x.strip()]
            if "all" not in regions:
                pq.regions = sorted(set(pq.regions) | set(regions))
    for n in (3, 2, 1):                                     # longest n-gram first
        for i in range(len(tokens) - n + 1):
            gram = " ".join(tokens[i:i + n])
            if len(gram) < 4 or all(t in STOPWORDS for t in tokens[i:i + n]):
                continue
            hit = process.extractOne(gram, PLACE_NAMES, scorer=fuzz.ratio, score_cutoff=FUZZY_THRESHOLD)
            if not hit:
                continue
            entry = PLACE_ENTRIES[hit[2]]
            if entry["kind"] == "spot" and entry["key"] not in pq.named_spots:
                pq.named_spots.append(entry["key"])
            elif entry["kind"] == "place" and pq.place is None:
                pq.place, pq.place_latlon = entry["key"], (float(entry["lat"]), float(entry["lon"]))


def parse_time(text, ref):
    """Returns (list of (start, end), label, found)."""
    day0 = ref.normalize()
    days, label = None, ""
    if re.search(r"\b(right now|now|currently|at the moment)\b", text):
        return [(ref, ref + pd.Timedelta(hours=3))], "next 3 hours", True
    if re.search(r"\bday after tomorrow\b", text):
        days, label = [day0 + pd.Timedelta(days=2)], "day after tomorrow"
    elif re.search(r"\btomorrow\b", text):
        days, label = [day0 + pd.Timedelta(days=1)], "tomorrow"
    elif re.search(r"\b(today|tonight|this (morning|afternoon|evening))\b", text):
        days, label = [day0], "today"
    elif re.search(r"\bnext weekend\b", text):
        sat = day0 + pd.Timedelta(days=(5 - ref.weekday()) % 7 + 7)
        days, label = [sat, sat + pd.Timedelta(days=1)], "next weekend"
    elif re.search(r"\bweekend\b", text):
        sat = day0 + pd.Timedelta(days=(5 - ref.weekday()) % 7)
        days = [d for d in [sat, sat + pd.Timedelta(days=1)] if d >= day0] if ref.weekday() != 6 else [day0]
        label = "this weekend"
    elif m := re.search(r"\bnext (\d+) days\b", text):
        days, label = [day0 + pd.Timedelta(days=i) for i in range(int(m.group(1)) + 1)], f"next {m.group(1)} days"
    elif re.search(r"\b(this week|next few days|coming days|this coming week)\b", text):
        days, label = [day0 + pd.Timedelta(days=i) for i in range(7)], "next 7 days"
    elif m := re.search(r"\b(" + "|".join(WEEKDAYS) + r")\b", text):
        ahead = (WEEKDAYS.index(m.group(1)) - ref.weekday()) % 7
        days, label = [day0 + pd.Timedelta(days=ahead)], m.group(1).capitalize()
    elif m := re.search(MONTH_PATTERN, text):
        parsed = dateparser.parse(m.group(0), settings={"RELATIVE_BASE": ref.to_pydatetime(), "PREFER_DATES_FROM": "future"})
        if parsed:
            days, label = [pd.Timestamp(parsed).normalize()], f"{parsed:%a %d %b}"
    part = next(((p, h0, h1) for p, h0, h1 in PARTS_OF_DAY if re.search(rf"\b{p}\b", text)), None)
    if days is None and part:
        days = [day0 if ref.hour < part[2] else day0 + pd.Timedelta(days=1)]
    if days is None:
        return [(ref, ref + pd.Timedelta(hours=DEFAULT_WINDOW_HOURS))], f"next {DEFAULT_WINDOW_HOURS} h (default)", False
    h0, h1 = (part[1], part[2]) if part else (0, 24)
    windows = [(d + pd.Timedelta(hours=h0), d + pd.Timedelta(hours=h1)) for d in days]
    windows = [(max(s, ref.floor("h")), e) for s, e in windows if e > ref]          # never recommend the past
    label = f"{label} {part[0]}".strip() if part else label
    return windows, label, True


def parse_query(query, ref_time):
    text = normalize_query(query)
    pq = ParsedQuery(raw=query, text=text)
    parse_location(text, pq)
    for skill, pattern in SKILL_PATTERNS.items():
        if re.search(pattern, text):
            pq.skill, pq.skill_from_query = skill, True
            break
    pq.windows, pq.time_label, pq.time_from_query = parse_time(text, pd.Timestamp(ref_time))
    if not pq.windows:
        pq.notes.append("The requested time has already passed in this snapshot.")
    pq.condition_intent = bool(pq.time_from_query or pq.skill_from_query or re.search(CONDITION_PATTERN, text))
    return pq


def default_parse(query, ref_time):
    """What the system 'understands' when query understanding is switched off (ablation)."""
    ref = pd.Timestamp(ref_time)
    return ParsedQuery(raw=query, text=normalize_query(query), condition_intent=True,
                       windows=[(ref, ref + pd.Timedelta(hours=DEFAULT_WINDOW_HOURS))],
                       time_label=f"next {DEFAULT_WINDOW_HOURS} h (default)")


def describe_parsed(pq):
    loc = pq.place or (", ".join(SPOTS[s]["name"] for s in pq.named_spots) if pq.named_spots else None) \
        or (" / ".join(pq.regions) if pq.regions else "all Taiwan")
    skill = pq.skill + ("" if pq.skill_from_query else " (default)")
    return f"Location: {loc} · Skill: {skill} · Time: {pq.time_label}"


if not LIBRARY_MODE:
    _ref = load_snapshot()["ref_time"]
    for q in ["beginner-friendly beach break near Gaoxiong this weekend",
              "uncrowded reef break on the east coast tomorrow morning", "where are the best waves?"]:
        print(f"{q!r}\n   -> {describe_parsed(parse_query(q, _ref))}")

## 3. Text relevance: BM25 with optional jargon expansion
Expansion adds the stems of matching synonym entries with weight < 1 (default 0.5). BM25 scores are then summed per stem and weighted. Scores are divided by the maximum over the corpus, so the text score lies in [0, 1] and can be combined with the condition-fit score.

In [ ]:
def query_stems(pq, expand):
    weights = {s: 1.0 for s in analyze(pq.text)}
    added = []
    if expand:
        for phrase, exps in SYNONYMS.items():
            if re.search(rf"\b{re.escape(phrase)}\b", pq.text):
                for e in exps:
                    for s in e["stems"]:
                        if weights.get(s, 0) < e["weight"]:
                            weights[s] = e["weight"]
                    added.append(e["expansion"])
    return weights, added


def bm25_scores(weights):
    bm25 = BM25_INDEX["bm25"]
    scores = np.zeros(len(DOC_IDS))
    for stem, w in weights.items():
        scores += w * bm25.get_scores([stem])
    return scores


def dense_scores(text):
    emb, ids = DENSE
    sims = emb @ embed_query(text)
    order = {d: i for i, d in enumerate(ids)}
    return np.array([sims[order[d]] for d in DOC_IDS])


def rrf(*score_arrays):
    fused = np.zeros(len(DOC_IDS))
    for scores in score_arrays:
        ranks = (-scores).argsort().argsort() + 1
        fused += 1.0 / (RRF_K + ranks)
    return fused


def matched_stems(weights, doc_id):
    doc = set(BM25_INDEX["doc_tokens"][DOC_IDS.index(doc_id)])
    return sorted(s for s in weights if s in doc)

## 4. The search function
**Ranking (improved system):**
1. *Filter* by location if the query names one (radius around a town, or a region). If nothing remains, the filter is relaxed and the user is told so; an empty result list is never useful.
2. *Text score* `T` ∈ [0, 1] (BM25 + expansion; a spot named in the query gets T = 1).
3. *Condition fit* `CF` ∈ [0, 1] for the user's skill and time window (best hour; for multi-day windows the best day).
4. *Fusion:* `score = α·T + (1 − α)·CF`, with `α = ALPHA_NO_CONDITION_INTENT` when the query does not ask about conditions.
5. *Safety:* spots too big for the user's level, or not suited to it, are moved below all others and carry a warning.

We chose a **weighted sum after hard filters** over learning-to-rank: ~50 judged queries are far too few to train an LTR model without overfitting, and a single interpretable α can be tuned on dev queries and explained in one sentence.

In [ ]:
SYSTEMS = {
    "baseline":        {"qu": False, "syn": False, "cf": False, "dense": False, "oracle": False},
    "improved":        {"qu": True,  "syn": True,  "cf": True,  "dense": False, "oracle": False},
    "improved_no_cf":  {"qu": True,  "syn": True,  "cf": False, "dense": False, "oracle": False},
    "improved_no_qu":  {"qu": False, "syn": True,  "cf": True,  "dense": False, "oracle": False},
    "improved_no_syn": {"qu": True,  "syn": False, "cf": True,  "dense": False, "oracle": False},
    "improved_hybrid": {"qu": True,  "syn": True,  "cf": True,  "dense": True,  "oracle": False},
    "improved_oracle": {"qu": True,  "syn": True,  "cf": True,  "dense": False, "oracle": True},
}
AVAILABLE_SYSTEMS = [s for s in SYSTEMS if not (SYSTEMS[s]["dense"] and DENSE is None)]


def _best_condition_fit(spot_id, snapshot, skill, windows, ref_time, use_observations):
    best = None
    for start, end in windows:
        r = condition_fit(spot_id, snapshot, skill, start, end, ref_time=ref_time, use_observations=use_observations)
        if best is None or r["cf"] > best["cf"]:
            best = r
    return best


def search_verbose(query, system="improved", k=DEFAULT_K, snapshot_date=None, skill=None, day=None,
                   snapshot=None, alpha=None, ref_time=None):
    """Full search with diagnostics. `snapshot` (a dict from load_snapshot / a live fetch) overrides `snapshot_date`.
    `skill` and `day` ("YYYY-MM-DD") override what query understanding found (used by the demo's selectors)."""
    t0 = time.perf_counter()
    cfg = SYSTEMS[system]
    snap = snapshot or load_snapshot(snapshot_date)
    ref = pd.Timestamp(ref_time) if ref_time is not None else snap["ref_time"]

    pq = parse_query(query, ref) if cfg["qu"] else default_parse(query, ref)
    if skill:
        pq.skill, pq.skill_from_query = skill, True
    if day:
        d = pd.Timestamp(day).normalize()
        pq.windows, pq.time_label, pq.time_from_query = [(max(d, ref.floor("h")), d + pd.Timedelta(days=1))], f"{d:%a %d %b}", True
        pq.condition_intent = True
    t1 = time.perf_counter()

    weights, expansions = query_stems(pq, expand=cfg["syn"])
    raw = bm25_scores(weights)
    if cfg["dense"]:
        text = rrf(raw, dense_scores(pq.text))
        text = text / text.max()
    else:
        text = raw / raw.max() if raw.max() > 0 else raw
    for sid in pq.named_spots:
        text[DOC_IDS.index(sid)] = 1.0

    keep = np.ones(len(DOC_IDS), dtype=bool)
    distance = np.full(len(DOC_IDS), np.nan)
    if cfg["qu"]:
        if pq.place_latlon:
            distance = _haversine_km(pq.place_latlon[0], pq.place_latlon[1], SPOT_LAT, SPOT_LON)
            keep = distance <= NEAR_RADIUS_KM
        elif pq.regions:
            keep = np.isin(SPOT_REGION, pq.regions)
        if pq.named_spots:
            keep |= np.isin(DOC_IDS, pq.named_spots)
        if not keep.any():
            pq.notes.append(f"No spots match the location filter; showing all spots{' (nearest first)' if pq.place_latlon else ''}.")
            keep[:] = True
    t2 = time.perf_counter()

    results = []
    for i in np.flatnonzero(keep):
        sid = DOC_IDS[i]
        row = {"spot_id": sid, "name": SPOTS[sid]["name"], "region": SPOTS[sid]["region"],
               "break_type": SPOTS[sid]["break_type"], "text_score": float(text[i]), "bm25": float(raw[i]),
               "distance_km": None if np.isnan(distance[i]) else round(float(distance[i]), 1),
               "matched_stems": matched_stems(weights, sid), "cf": None, "verdict": "", "best_time": None,
               "surfable_hours": "", "explanation": "", "warning": "",
               "factors": {}, "conditions": {}, "weights": {}, "alpha": None, "safety_penalty": 0.0, "distance_penalty": 0.0}
        if system == "baseline":
            row["score"] = float(raw[i])
        else:
            if pq.windows:
                cf = _best_condition_fit(sid, snap, pq.skill, pq.windows, ref, cfg["oracle"])
                row.update({k: cf[k] for k in ["cf", "verdict", "best_time", "surfable_hours", "explanation", "factors"]})
                row.update({k: cf.get(k, {}) for k in ["conditions", "weights"]})
                unsafe = cf["too_big"] or cf["factors"].get("f_skill", 1) == 0
                if unsafe:
                    row["warning"] = f"Not recommended for {SKILL_PLURAL[pq.skill]}"
            else:
                unsafe = False
            if cfg["cf"] and row["cf"] is not None:
                a = alpha if alpha is not None else (ALPHA if pq.condition_intent else ALPHA_NO_CONDITION_INTENT)
                row["score"] = a * row["text_score"] + (1 - a) * row["cf"]
                row["alpha"] = a
                if SAFETY_DEMOTE and unsafe:
                    row["score"] -= 1.0
                    row["safety_penalty"] = -1.0
            else:
                row["score"] = row["text_score"]
            if pq.place_latlon and not np.isnan(distance[i]) and not np.any(keep & (distance <= NEAR_RADIUS_KM)):
                row["score"] -= distance[i] / 1000.0          # relaxed location filter: nearer spots first on ties
                row["distance_penalty"] = -distance[i] / 1000.0
        results.append(row)
    t3 = time.perf_counter()

    results.sort(key=lambda r: (-r["score"], r["spot_id"]))
    results = results[:k]
    for rank, r in enumerate(results, 1):
        r["rank"] = rank
        r["best_time"] = None if r["best_time"] is None or pd.isna(r["best_time"]) else f"{r['best_time']:%a %d %b %H:%M}"
    return {"query": query, "system": system, "results": results, "parsed": pq, "expansions": expansions,
            "snapshot": snap["key"], "ref_time": ref,
            "latency_ms": {"understanding": (t1 - t0) * 1000, "text": (t2 - t1) * 1000,
                           "conditions": (t3 - t2) * 1000, "total": (time.perf_counter() - t0) * 1000}}


def search(query, system="improved", k=DEFAULT_K, snapshot_date=None, **kwargs):
    """The shared interface: returns a ranked list of result dicts (rank, spot_id, name, score, explanation, ...)."""
    return search_verbose(query, system=system, k=k, snapshot_date=snapshot_date, **kwargs)["results"]

### Highlighting matched terms
Used by the demo: words in a profile whose stem matched a (possibly expanded) query stem are wrapped in `<mark>`. This shows users *why* a spot was retrieved, and it shows the audience the effect of expansion.

In [ ]:
def highlight(text, stems):
    stems = set(stems)
    out = []
    for piece in re.split(r"(\W+)", str(text)):
        word = piece.lower()
        if word and re.fullmatch(r"\w+", word) and word not in STOPWORDS and _stemmer.stem(word) in stems:
            out.append(f"<mark>{escape(piece)}</mark>")
        else:
            out.append(escape(piece))
    return "".join(out)

## 5. Examples: baseline vs improved, side by side
These queries are only for a first visual check (with the placeholder corpus, the spot names are not meaningful). The systematic comparison is in `05_evaluation`.

In [ ]:
EXAMPLE_QUERIES = [
    "beginner-friendly beach break near Kaohsiung this weekend",
    "uncrowded reef break on the east coast tomorrow morning",
    "glassy waves in Yilan on Saturday",
    "where are the best waves this weekend?",
    "point break with board rental near Kenting",
]


def side_by_side(query, k=3, snapshot_date=None):
    b = search_verbose(query, "baseline", k, snapshot_date)
    i = search_verbose(query, "improved", k, snapshot_date)
    left = pd.DataFrame([{"spot": r["name"], "bm25": round(r["score"], 2)} for r in b["results"]])
    right = pd.DataFrame([{"spot": r["name"], "score": round(r["score"], 2), "CF": round(r["cf"] or 0, 2),
                           "verdict": r["verdict"], "best": r["best_time"]} for r in i["results"]])
    table = pd.concat({"Baseline (BM25)": left, "Improved (condition-aware)": right}, axis=1)
    table.index = table.index + 1
    return table, i


if not LIBRARY_MODE:
    for q in EXAMPLE_QUERIES:
        table, info = side_by_side(q)
        display(HTML(f"<h4>{escape(q)}</h4><p><i>{escape(describe_parsed(info['parsed']))}</i>"
                     f"{'<br>' + escape(' '.join(info['parsed'].notes)) if info['parsed'].notes else ''}</p>"))
        display(table)
        if info["results"]:
            print("Top result explanation:", info["results"][0]["explanation"])